In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql import Window

# ---------------------------------------------------------------------------
# Q1: mean/stddev of annual US population, 2013-2018
# (PySpark alternative)
# ---------------------------------------------------------------------------

@dp.materialized_view(name="rearc_bls.gold.gold_population_stats_pyspark")
def gold_population_stats_pyspark():
    df = dp.read("rearc_bls.silver.silver_population").filter(F.col("year").between(2013, 2018))
    return df.select(
        F.round(F.mean("population"), 2).alias("mean_population"),
        F.round(F.stddev("population"), 2).alias("stddev_population"),
    )


# ---------------------------------------------------------------------------
# Q2: For every series_id in the BLS data, what's the best year, meaning the year with the largest sum of value across all its quarters?
# (PySpark primary -- uses a Window function for the per-series ranking)
# ---------------------------------------------------------------------------

@dp.materialized_view(name="rearc_bls.gold.gold_best_year_per_series")
def gold_best_year_per_series():
    yearly = (
        dp.read("rearc_bls.silver.silver_bls_pr")
        .groupBy("series_id", "year")
        .agg(F.sum("value").alias("total_value"))
    )

    w = Window.partitionBy("series_id").orderBy(F.col("total_value").desc(), F.col("year").asc())
    best = (
        yearly
        .withColumn("rn", F.row_number().over(w))
        .filter("rn = 1")
        .drop("rn")
    )

    series = dp.read("rearc_bls.silver.silver_bls_series")
    sector = dp.read("rearc_bls.silver.silver_bls_sector").select("sector_code", "sector_name")
    cls = dp.read("rearc_bls.silver.silver_bls_class").select("class_code", "class_text")
    measure = dp.read("rearc_bls.silver.silver_bls_measure").select("measure_code", "measure_text")
    duration = dp.read("rearc_bls.silver.silver_bls_duration").select("duration_code", "duration_text")

    decoded = (
        series
        .join(sector, "sector_code", "left")
        .join(cls, "class_code", "left")
        .join(measure, "measure_code", "left")
        .join(duration, "duration_code", "left")
        .withColumn(
            "human_label",
            F.concat_ws(
                " - ",
                F.col("sector_name"),
                F.col("class_text"),
                F.col("measure_text"),
                F.col("duration_text"),
            ),
        )
        .select("series_id", "human_label")
    )

    return (
        best.join(decoded, "series_id", "left")
        .select("series_id", "human_label", "year", "total_value")
    )


# ---------------------------------------------------------------------------
# For series_id = PRS30006032 and period = Q01, what was the value each year, joined with that year's population where available?
# (PySpark alternative)
# ---------------------------------------------------------------------------

@dp.materialized_view(name="rearc_bls.gold.gold_prs30006032_q01_pyspark")
def gold_prs30006032_q01_pyspark():
    b = dp.read("rearc_bls.silver.silver_bls_pr").filter(
        "series_id = 'PRS30006032' AND period = 'Q01'"
    )
    p = dp.read("rearc_bls.silver.silver_population")
    return (
        b.join(p, "year", "left")
        .select("year", "value", "population")
        .orderBy("year")
    )